# IR4 + 7B QLoRA — 完整数据版 v3
独立训练包已内置五折 IR8 缓存、全部训练 QA、原始分折和 test/pilot 缓存。
不需要另挂载五折缓存；只需训练包、云端 CUDA GPU 和已验证权重或下载网络。
模型 revision、代码、训练配置和依赖锁均固定在包内，原 baseline Notebook 与结果不受影响。
数据/哈希/CPU 验证通过不代表真实 GPU 训练通过；先执行短跑及 adapter 重载验证。

Create the package locally and copy its printed `manifest_sha256` into `EXPECTED_MANIFEST_SHA256` before extraction. Attach only that exact private package to Kaggle.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, re, shutil, subprocess, sys, tempfile, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
BUNDLE_INPUT = None
EXPECTED_MANIFEST_SHA256 = None  # paste manifest_sha256 from the trusted package command  # ZIP 文件，或含 training_bundle_manifest.json 的已解压目录
WEIGHTS_INPUT = None  # 可选：含原 cuhkx_weights.json 和全部权重文件的目录
EXPERIMENT = "sft_full_v3"  # 新配置使用新的实验名/训练包
GPU = 0
RUN_CONFIRMATION = False  # 开发集选定候选后开启
RUN_TEST = False          # 确认有收益后开启


## 1. 验证独立训练包并准备工作副本（只用标准库）


In [ ]:
import hmac, stat

MAX_ARCHIVE_BYTES = 512 * 1024**2
MAX_ARCHIVE_FILES = 20_000
MAX_EXPANDED_BYTES = 512 * 1024**2
MAX_MEMBER_BYTES = 16 * 1024**2
MAX_COMPRESSION_RATIO = 50.0
FREE_SPACE_RESERVE = 256 * 1024**2

if not isinstance(EXPECTED_MANIFEST_SHA256, str) or re.fullmatch(r"[0-9a-f]{64}", EXPECTED_MANIFEST_SHA256) is None:
    raise RuntimeError("set EXPECTED_MANIFEST_SHA256 from the trusted local package command")


def _safe_bundle_name(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name
            or path.as_posix() != name or any(ord(character) < 32 for character in name)):
        raise RuntimeError("unsafe package path: " + repr(name))
    return path


def _open_bounded_archive(bundle):
    if not bundle.is_file():
        return None, None, 0
    if bundle.stat().st_size > MAX_ARCHIVE_BYTES:
        raise RuntimeError("package exceeds compressed-size budget")
    archive = zipfile.ZipFile(bundle)
    infos = archive.infolist()
    if not infos or len(infos) > MAX_ARCHIVE_FILES:
        raise RuntimeError("package entry count is outside the allowed budget")
    index, folded, expanded = {}, set(), 0
    for info in infos:
        _safe_bundle_name(info.filename)
        folded_name = info.filename.casefold()
        if info.filename in index or folded_name in folded:
            raise RuntimeError("package contains duplicate or case-colliding paths")
        mode = (info.external_attr >> 16) & 0xFFFF
        if (info.is_dir() or info.flag_bits & 1 or stat.S_IFMT(mode) not in (0, stat.S_IFREG)
                or info.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)):
            raise RuntimeError("package contains an unsupported entry")
        if info.file_size < 0 or info.file_size > MAX_MEMBER_BYTES:
            raise RuntimeError("package member exceeds size budget")
        if info.file_size and (not info.compress_size
                or info.file_size / info.compress_size > MAX_COMPRESSION_RATIO):
            raise RuntimeError("package member exceeds compression-ratio budget")
        expanded += info.file_size
        if expanded > MAX_EXPANDED_BYTES:
            raise RuntimeError("package exceeds expanded-size budget")
        index[info.filename] = info
        folded.add(folded_name)
    return archive, index, expanded


def _directory_member(bundle, name):
    source = bundle / name
    if source.is_symlink():
        raise RuntimeError("package symlinks are unsupported: " + name)
    path = source.resolve()
    if not path.is_relative_to(bundle.resolve()) or not path.is_file():
        raise RuntimeError("unsafe package file: " + name)
    if path.stat().st_size > MAX_MEMBER_BYTES:
        raise RuntimeError("package member exceeds size budget")
    return path


def _member_bytes(bundle, archive, index, name):
    if archive:
        info = index.get(name)
        if info is None:
            raise RuntimeError("package member is missing: " + name)
        with archive.open(info) as handle:
            content = handle.read(MAX_MEMBER_BYTES + 1)
        if len(content) != info.file_size or len(content) > MAX_MEMBER_BYTES:
            raise RuntimeError("package member size changed while reading")
        return content
    return _directory_member(bundle, name).read_bytes()


def _trusted_manifest(bundle, archive, index, marker):
    raw = _member_bytes(bundle, archive, index, marker)
    digest = hashlib.sha256(raw).hexdigest()
    if not hmac.compare_digest(digest, EXPECTED_MANIFEST_SHA256):
        raise RuntimeError("package manifest is not the trusted release")
    return raw, json.loads(raw)


def _validated_bundle_entries(bundle, archive, index, marker, manifest, prefix):
    entries = manifest.get("files")
    if not isinstance(entries, list) or len(entries) > MAX_ARCHIVE_FILES - 1:
        raise RuntimeError("invalid package file list")
    expected, folded, total = {}, set(), 0
    for entry in entries:
        if not isinstance(entry, dict) or set(entry) != {"path", "bytes", "sha256"}:
            raise RuntimeError("invalid package entry")
        name, size, digest = entry["path"], entry["bytes"], entry["sha256"]
        path = _safe_bundle_name(name)
        if (not name.startswith(prefix) or not isinstance(size, int) or isinstance(size, bool)
                or size < 0 or size > MAX_MEMBER_BYTES or not isinstance(digest, str)
                or re.fullmatch(r"[0-9a-f]{64}", digest) is None):
            raise RuntimeError("invalid package path, size, or digest")
        if name in expected or name.casefold() in folded:
            raise RuntimeError("package manifest contains duplicate paths")
        if archive:
            if name not in index or index[name].file_size != size:
                raise RuntimeError("package manifest differs from ZIP metadata")
        else:
            if _directory_member(bundle, name).stat().st_size != size:
                raise RuntimeError("package manifest differs from directory metadata")
        expected[name] = entry
        folded.add(name.casefold())
        total += size
        if total > MAX_EXPANDED_BYTES:
            raise RuntimeError("package manifest exceeds expanded-size budget")
    actual = set(index) if archive else {marker, *expected}
    if actual != set(expected) | {marker}:
        raise RuntimeError("package file set differs from manifest")
    disk_root = WORK
    while not disk_root.exists():
        disk_root = disk_root.parent
    if shutil.disk_usage(disk_root).free < total + FREE_SPACE_RESERVE:
        raise RuntimeError("insufficient free space for bounded extraction")
    return list(expected.values())


def _verify_entry(bundle, archive, index, entry):
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    with source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            size += len(block)
            if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                raise RuntimeError("package member exceeded manifest size")
            digest.update(block)
    if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
        raise RuntimeError("package content hash mismatch: " + entry["path"])


def _copy_entry(bundle, archive, index, entry, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".partial")
    if partial.exists():
        partial.unlink()
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    try:
        with source, partial.open("xb") as output:
            for block in iter(lambda: source.read(1024 * 1024), b""):
                size += len(block)
                if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                    raise RuntimeError("package member exceeded manifest size")
                digest.update(block)
                output.write(block)
        if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
            raise RuntimeError("package content changed while copying")
        os.replace(partial, target)
    finally:
        if partial.exists():
            partial.unlink()

PACKAGE_ID = 'cuhkx-ir4-qlora-full-v3'
MARKER = 'training_bundle_manifest.json'
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", EXPERIMENT):
    raise RuntimeError("invalid experiment name")
if BUNDLE_INPUT is None:
    candidates = []
    for candidate_marker in INPUT.rglob(MARKER):
        try:
            if candidate_marker.stat().st_size <= MAX_MEMBER_BYTES:
                value = json.loads(candidate_marker.read_text(encoding="utf-8"))
                if value.get('package_id') == PACKAGE_ID:
                    candidates.append(candidate_marker.parent)
        except (OSError, ValueError):
            pass
    if not candidates:
        candidates = list(INPUT.rglob('cuhkx-ir4-qlora-full-v3.zip'))
    if len(candidates) != 1:
        raise RuntimeError(f"found {len(candidates)} matching packages; set BUNDLE_INPUT")
    BUNDLE_INPUT = candidates[0]
BUNDLE_INPUT = Path(BUNDLE_INPUT).resolve()
archive, archive_index, expanded_bytes = _open_bounded_archive(BUNDLE_INPUT)
try:
    raw_manifest, manifest = _trusted_manifest(
        BUNDLE_INPUT, archive, archive_index, MARKER)
    if manifest.get("schema_version") != 1 or manifest.get('package_id') != PACKAGE_ID:
        raise RuntimeError("wrong package identity")
    if manifest.get("training_cache_mode") != "embedded_complete":
        raise RuntimeError("training package does not contain the complete cache")
    entries = _validated_bundle_entries(
        BUNDLE_INPUT, archive, archive_index, MARKER, manifest, 'training_repo/')
    for entry in entries:
        _verify_entry(BUNDLE_INPUT, archive, archive_index, entry)
    MANIFEST_SHA256 = hashlib.sha256(raw_manifest).hexdigest()
    WORK_ROOT = WORK.resolve()
    RUNTIME = WORK_ROOT / ('cuhkx_qlora_' + MANIFEST_SHA256[:12]) / EXPERIMENT
    if RUNTIME.is_symlink():
        raise RuntimeError("runtime root must not be a symlink")
    RUNTIME_ROOT = RUNTIME.resolve()
    if not RUNTIME_ROOT.is_relative_to(WORK_ROOT):
        raise RuntimeError("runtime root escapes working directory")
    for entry in entries:
        candidate = RUNTIME_ROOT / entry["path"]
        target = candidate.resolve()
        if candidate.is_symlink() or not target.is_relative_to(RUNTIME_ROOT):
            raise RuntimeError("runtime path escapes package root")
        if target.exists():
            if target.is_symlink() or not target.is_file():
                raise RuntimeError("runtime contains an unsafe existing path")
            digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if target.stat().st_size != entry["bytes"] or digest != entry["sha256"]:
                raise RuntimeError("runtime copy was modified; use a new experiment/runtime")
    for entry in entries:
        target = (RUNTIME_ROOT / entry["path"]).resolve()
        if not target.exists():
            _copy_entry(BUNDLE_INPUT, archive, archive_index, entry, target)
    REPO = RUNTIME_ROOT / 'training_repo'
    (RUNTIME_ROOT / MARKER).write_bytes(raw_manifest)
finally:
    if archive:
        archive.close()
print("Verified repository:", REPO)
print("Trusted manifest SHA256:", MANIFEST_SHA256)
print("Cache mode:", manifest["training_cache_mode"])
print("Pinned revision:", manifest["model_revision"])


## 2. 确认包内完整缓存，无需复制外挂数据


In [ ]:
if manifest.get("training_cache_mode") != "embedded_complete":
    raise RuntimeError("请选择完整数据版 ZIP")
coverage = manifest.get("training_coverage", {})
if set(coverage) != {"train", "dev", "confirm"} or any(c["missing_qa"] for c in coverage.values()):
    raise RuntimeError("训练包声明的数据不完整")
print("包内五折缓存:", manifest.get("training_cache_summary"))
print("分组:", coverage)
print("下一步独立核验全部图片、QA 关联与数据指纹。")


## 3. 独立 Python 3.11 环境与 CPU 数据检查


In [ ]:
VENV = RUNTIME / "train_env"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    with tempfile.TemporaryDirectory(prefix="cuhkx_uv_bootstrap_",dir=WORK) as bootstrap_dir:
        BOOT = Path(bootstrap_dir)
        subprocess.run([sys.executable,"-m","pip","install","--target",str(BOOT),"--no-deps","--require-hashes","--only-binary=:all:","-r",str(REPO/"requirements/bootstrap.lock.txt")],check=True)
        subprocess.run([sys.executable,"-m","uv","venv","--python","3.11","--seed",str(VENV)],
                       env={**os.environ,"PYTHONPATH":str(BOOT)},check=True)
subprocess.run([str(PYTHON),"-m","pip","install","--require-hashes","--only-binary=:all:","-r",str(REPO/"requirements/cpu.lock.txt")],check=True)
subprocess.run([str(PYTHON),"-m","pip","install","--no-deps","--no-build-isolation","-e",str(REPO)],check=True)

def command(*args):
    return [str(PYTHON),"-m","cuhkx.cli",*args,"--project-root",str(REPO)]

CLOUD_ENV = {**os.environ,"PYTHONPATH":str(REPO/"src"),"PYTHONDONTWRITEBYTECODE":"1"}

def cloud(*args):
    from collections import deque
    tail = deque(maxlen=80)
    env = {**CLOUD_ENV,"PYTHONUNBUFFERED":"1","CUHKX_TRACEBACK":"1"}
    with subprocess.Popen(command(*args),cwd=REPO,env=env,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,encoding="utf-8",errors="replace",bufsize=1) as process:
        for line in process.stdout:
            print(line,end="",flush=True)
            tail.append(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"{args[0]} exited with code {code}; last output:\n" + "".join(tail))

data_state = json.loads(subprocess.check_output(command("training-check"),cwd=REPO,env=CLOUD_ENV,text=True))
if data_state["status"] != "PASS": raise RuntimeError("五折缓存尚未完整")
if manifest["training_data_signature"] is not None and data_state["data_signature"] != manifest["training_data_signature"]:
    raise RuntimeError("完整数据包的数据指纹发生变化")
print(json.dumps(data_state["coverage"],indent=2))


## 4. 安装固定训练依赖；只在云端 CUDA 环境执行


In [ ]:
subprocess.run([str(PYTHON),"-m","pip","install","--require-hashes","--only-binary=:all:","-r",str(REPO/"requirements/train.lock.txt")],check=True)
subprocess.run([str(PYTHON),"-m","pip","check"],check=True)
probe = "import json,sys,torch; assert sys.version_info[:2]==(3,11); assert torch.cuda.is_available(), '需要云端 CUDA GPU'; print(json.dumps({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))"
environment = subprocess.check_output([str(PYTHON),"-c",probe],text=True)
(RUNTIME/"environment.json").write_text(environment)
(RUNTIME/"environment.freeze.txt").write_text(subprocess.check_output([str(PYTHON),"-m","pip","freeze","--all"],text=True))
print(environment)


## 5. 准备与 baseline 同一 revision 的权重，不使用浮动 main


In [ ]:
expected_receipt = json.loads((REPO/"provenance/base_weights.json").read_text())
revision = manifest["model_revision"]
if not re.fullmatch(r"[0-9a-f]{40}", revision): raise RuntimeError("包内 revision 未锁定")
if WEIGHTS_INPUT is None:
    candidates = []
    for receipt_path in INPUT.rglob("cuhkx_weights.json"):
        candidate = json.loads(receipt_path.read_text())
        if candidate.get("model_id") == manifest["model_id"] and candidate.get("revision") == revision:
            if candidate == expected_receipt and all((receipt_path.parent/e["path"]).is_file() for e in candidate["files"]):
                candidates.append(receipt_path.parent)
    if len(candidates)>1: raise RuntimeError("找到多份匹配权重，请设置 WEIGHTS_INPUT")
    WEIGHTS = candidates[0] if candidates else Path("/tmp")/("cuhkx_qlora_weights_"+revision[:12])
else:
    WEIGHTS = Path(WEIGHTS_INPUT)
if not (WEIGHTS/"cuhkx_weights.json").exists():
    needed = sum(e["bytes"] for e in expected_receipt["files"]) + 2*1024**3
    if shutil.disk_usage(WEIGHTS.parent if WEIGHTS.parent.exists() else Path("/tmp")).free < needed:
        raise RuntimeError("权重存储空间不足，请挂载完整的已验证权重目录")
cloud("fetch-weights","--weights-dir",str(WEIGHTS))
if json.loads((WEIGHTS/"cuhkx_weights.json").read_text()) != expected_receipt:
    raise RuntimeError("权重与原 baseline 的来源清单不同")
(RUNTIME/"session.json").write_text(json.dumps({"package_manifest_sha256":PACKAGE_SHA,"training_data_signature":data_state["data_signature"],"model_revision":revision,"experiment":EXPERIMENT},indent=2))


## 6. 短跑与 adapter 重载验证；这不是正式性能结果


In [ ]:
cloud("train","--weights-dir",str(WEIGHTS),"--run-id","pt_smoke","--smoke-steps","4","--gpu",str(GPU),"--resume")
SMOKE_ADAPTER = REPO/"artifacts/training/pt_smoke/adapter"
cloud("predict","--dataset","pilot","--limit","16","--weights-dir",str(WEIGHTS),"--adapter-dir",str(SMOKE_ADAPTER),"--run-id","pt_smoke_reload","--resume")


## 7. 记录相同 dev 集合上的基座对照，然后正式训练


In [ ]:
cloud("evaluate-training","--split","dev","--weights-dir",str(WEIGHTS),"--run-id","pt_base_dev","--resume")
cloud("train","--weights-dir",str(WEIGHTS),"--run-id","pt_sft","--gpu",str(GPU),"--resume")
ADAPTER = REPO/"artifacts/training/pt_sft/adapter"
cloud("evaluate-training","--split","dev","--weights-dir",str(WEIGHTS),"--adapter-dir",str(ADAPTER),"--run-id","pt_adapter_dev","--resume")
def accuracy(run_id):
    return json.loads((REPO/"outputs"/run_id/"metrics.json").read_text())["metrics"]["overall_accuracy"]
print("dev baseline:",accuracy("pt_base_dev"),"adapter:",accuracy("pt_adapter_dev"))


## 8. 候选选定后才开启确认；本轮不据确认结果反复调参


In [ ]:
CONFIRMED = False
if RUN_CONFIRMATION:
    cloud("verify-run","--run-id","pt_base_dev")
    cloud("verify-run","--run-id","pt_adapter_dev")
    if accuracy("pt_adapter_dev") <= accuracy("pt_base_dev"):
        raise RuntimeError("开发集尚无提升，停止确认/测试生产")
    cloud("evaluate-training","--split","confirm","--weights-dir",str(WEIGHTS),"--run-id","pt_base_confirm","--resume")
    cloud("evaluate-training","--split","confirm","--weights-dir",str(WEIGHTS),"--adapter-dir",str(ADAPTER),"--run-id","pt_adapter_confirm","--resume")
    CONFIRMED = accuracy("pt_adapter_confirm") > accuracy("pt_base_confirm")
    print("confirm baseline:",accuracy("pt_base_confirm"),"adapter:",accuracy("pt_adapter_confirm"),"improved:",CONFIRMED)
else:
    print("确认阶段未开启。先固定配置与候选，再设置 RUN_CONFIRMATION。")


## 9. 有收益的候选单独导出测试提交；不自动上传


In [ ]:
if RUN_TEST:
    if not RUN_CONFIRMATION or not CONFIRMED:
        raise RuntimeError("确认集尚未显示提升，不生成正式测试候选")
    cloud("verify-run","--run-id","pt_adapter_confirm")
    cloud("predict","--dataset","test","--weights-dir",str(WEIGHTS),"--adapter-dir",str(ADAPTER),"--run-id","pt_test","--resume")
    cloud("submit","--run-id","pt_test")
    print("新提交:",REPO/"outputs/pt_test/submission.csv")
print("保存最终 adapter 与所需 checkpoint:",REPO/"artifacts/training")
print("保存真实预测与评测:",REPO/"outputs")
print("保存环境与训练包身份:",RUNTIME)
